# 07. Boxcar covariance filtering

Before beamforming, the pipeline estimates the sample covariance by spatial multilooking with a Boxcar window of size `filter_arguments['win'] = [az, rg]`. This notebook demonstrates the boxcar averaging operation on synthetic data and shows how the window size trades speckle reduction against spatial resolution in the estimated coherence, which directly affects beamforming quality.

**Modules exercised:** configuration.processing_config (TomogramConfiguration.filter_method, filter_arguments), pipelines.processing_pipeline.tomogram (filter stage)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## A two-track stack with a coherence edge

We build a scene whose left half is fully coherent between two tracks and whose right half is incoherent. A boxcar estimate of coherence should recover this structure, blurred by the window.

In [ ]:
n_az, n_rg = 80, 120
az = np.arange(n_az)[:, None]; rg = np.arange(n_rg)[None, :]

primary   = (RNG.standard_normal((n_az, n_rg)) + 1.0j * RNG.standard_normal((n_az, n_rg))).astype(np.complex64)
coherent  = primary.copy()
incoherent = (RNG.standard_normal((n_az, n_rg)) + 1.0j * RNG.standard_normal((n_az, n_rg))).astype(np.complex64)

mask      = (rg < n_rg // 2)
secondary = np.where(mask, coherent, incoherent).astype(np.complex64)
print('left half coherent, right half incoherent')

## Boxcar coherence estimator

Coherence is `|<p s*>| / sqrt(<|p|^2><|s|^2>)`, where `<.>` is a boxcar average. We implement the boxcar as a separable uniform convolution and vary the window.

In [ ]:
def boxcar(field, win):
    win_az, win_rg = win
    kernel_az = np.ones((win_az, 1)) / win_az
    kernel_rg = np.ones((1, win_rg)) / win_rg
    from scipy.signal import convolve2d
    smoothed = convolve2d(field, kernel_az, mode='same', boundary='symm')
    smoothed = convolve2d(smoothed, kernel_rg, mode='same', boundary='symm')
    return smoothed

def coherence(primary, secondary, win):
    numerator   = np.abs(boxcar(primary * np.conj(secondary), win))
    denominator = np.sqrt(boxcar(np.abs(primary) ** 2, win) * boxcar(np.abs(secondary) ** 2, win))
    return numerator / (denominator + 1e-30)

windows = [[3, 3], [9, 9], [20, 10]]
coh_maps = {tuple(w): coherence(primary, secondary, w) for w in windows}
print('windows evaluated:', list(coh_maps.keys()))

## Effect of window size

Larger windows suppress speckle in the coherence estimate (the left half approaches 1, the right half approaches 0) but blur the edge between the two regions.

In [ ]:
fig, axes = plt.subplots(1, len(windows), figsize=(4 * len(windows), 3.4))
for ax, w in zip(axes, windows):
    im = ax.imshow(coh_maps[tuple(w)], aspect='auto', origin='lower', vmin=0.0, vmax=1.0)
    ax.set_title(f'coherence, win={w}')
    ax.set_xlabel('range'); ax.set_ylabel('azimuth')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
fig.tight_layout()
plt.show()

## Mean coherence per region versus window size

Quantifies the separation: the coherent region should rise toward 1 and the incoherent region fall toward its bias floor as the window grows.

In [ ]:
win_sizes = [[1, 1], [3, 3], [5, 5], [9, 9], [15, 15], [20, 10]]
left, right = [], []
half = n_rg // 2
for w in win_sizes:
    c = coherence(primary, secondary, w)
    left.append(float(c[:, :half].mean()))
    right.append(float(c[:, half:].mean()))

looks = [w[0] * w[1] for w in win_sizes]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(looks, left,  marker='o', label='coherent region')
ax.plot(looks, right, marker='s', label='incoherent region')
ax.set_xlabel('window looks (az * rg)'); ax.set_ylabel('mean coherence')
ax.set_title('Coherence vs boxcar window size')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

Small windows give a noisy coherence map with high bias in the incoherent half; as the window grows the left half saturates near 1 and the right half drops, while the boundary blurs. The summary plot shows the two regions separating with increasing looks, confirming the resolution-versus-estimation-quality trade-off controlled by `filter_arguments['win']`.